In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



In [ ]:
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА ДАННЫХ
# ======================
digits = load_digits()
X, y = digits.data, digits.target
images = digits.images



In [ ]:
print("Number of samples:", X.shape[0])
print("Feature dim:", X.shape[1])
print("Image shape:", images.shape[1:])
print("Classes:", np.unique(y))



In [ ]:
pd.Series(y).value_counts().sort_index()



In [ ]:
# Подготовка данных: считаем частоту каждой цифры 0-9
class_counts = pd.Series(y).value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=class_counts.index.astype(str),
    y=class_counts.values,
    hue=class_counts.index.astype(str),
    palette="viridis",
    legend=False,
)
plt.title("Distribution of digit classes", fontsize=16, pad=16)
plt.xlabel("Digit", fontsize=12)
plt.ylabel("Number of samples", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Подписи над столбцами
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 3, str(count), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()



In [ ]:
# Сетка примеров рукописных цифр (8x8 пикселей)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Sample digits (one per class)", fontsize=16, y=1.02)

for digit, ax in enumerate(axes.flatten()):
    # Берём первое изображение нужного класса
    idx = np.where(y == digit)[0][0]
    ax.imshow(images[idx], cmap="gray_r")
    ax.set_title(f"Class: {digit}", fontsize=11)
    ax.axis("off")
    # Сохраняем отдельные PNG для портфолио
    plt.imsave(FIG / f"digit_sample_{digit}.png", images[idx], cmap="gray_r")

plt.tight_layout()
plt.show()



In [ ]:
# Монтаж случайных 30 изображений
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(images), size=30, replace=False)

fig, axes = plt.subplots(5, 6, figsize=(12, 10))
fig.suptitle("Random sample of digits", fontsize=18, y=1.01)

for ax in axes.flatten()[30:]:
    ax.axis("off")

for i, idx in enumerate(sample_idx):
    ax = axes[i // 6, i % 6]
    ax.imshow(images[idx], cmap="gray_r")
    ax.set_title(f"y={y[idx]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIG / "digits_montage.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# ======================
# 2. TRAIN / TEST SPLIT
# ======================
RANDOM_SEED = 0
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)



In [ ]:
# ======================
# 3. LOGISTIC REGRESSION
# ======================
logreg = LogisticRegression(max_iter=5000, random_state=RANDOM_SEED)
logreg.fit(X_train, y_train)
pred_lr = logreg.predict(X_test)
print("logreg accuracy:", round(accuracy_score(y_test, pred_lr), 4))



In [ ]:
# ======================
# 4. RANDOM FOREST
# ======================
rf = RandomForestClassifier(n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print("rf accuracy:", round(accuracy_score(y_test, pred_rf), 4))



In [ ]:
print("=== LogisticRegression ===")
print(classification_report(y_test, pred_lr, digits=3))
print("=== RandomForest ===")
print(classification_report(y_test, pred_rf, digits=3))



In [ ]:
# Матрица ошибок для RandomForest
cm = confusion_matrix(y_test, pred_rf)

plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (RandomForest)")
plt.tight_layout()
plt.show()



In [ ]:
# Сравнение точности двух моделей
scores = pd.Series({
    "LogisticRegression": accuracy_score(y_test, pred_lr),
    "RandomForest": accuracy_score(y_test, pred_rf),
})

plt.figure(figsize=(7, 4))
sns.barplot(x=scores.index, y=scores.values, palette="viridis")
plt.ylim(0.8, 1.0)
plt.ylabel("Accuracy")
plt.title("Model comparison on digits", fontsize=14, pad=12)
for i, v in enumerate(scores.values):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()



In [ ]:
# Несколько ошибочных предсказаний RF (если есть)
wrong = np.where(pred_rf != y_test)[0][:12]
if len(wrong) == 0:
    print("No errors on this split")
else:
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    fig.suptitle("Misclassified digits (RF)", fontsize=14)
    for ax, i in zip(axes.flatten(), wrong):
        # X_test строки — разворачиваем обратно в 8x8
        ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
        ax.set_title(f"true={y_test[i]} pred={pred_rf[i]}", fontsize=9, color="red")
        ax.axis("off")
    for ax in axes.flatten()[len(wrong):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

